# 🚦 16 · Bayesian networks II: reading independence from the graph
### Conditional independence, the three canonical triples, explaining away, Markov blankets and d-separation — all 15 worked examples of the slides, checked by brute force and tested on real signalling data

**BiologyToML** · Part D — Probabilistic graphical models · Module 16 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/16_bayesian_networks_independence.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib (graphs drawn with the toolkit's `draw_graph`) |
| **Prerequisites** | Module 15 (DAGs, CPTs, the factorized joint distribution) and Module 02 (conditional probability) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | `008_BN_mz_021825` slides 38–96 — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The alarm-network CPTs are those of the slides. CPTs on the seven-node d-separation graph and on the small
> biological graphs are random or illustrative (SIMULATED), chosen only to test what the graphs predict.
> Section 7 tests two predictions of the Sachs et al. (2005) consensus network on the real single-cell data
> (1,800 unperturbed cells, discretized to low/medium/high).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Define** independence and conditional independence in two equivalent ways and **test** them on a joint table — exactly, and with the conditional mutual information.
2. **Explain** the three canonical triples (causal chain, common cause, common effect) and **predict** when observing the middle node blocks or opens the flow of information; **compute** explaining away in the alarm network.
3. **Identify** the Markov blanket of any node (parents, children, co-parents), **derive** why it suffices, and **verify** numerically that $P(X\mid\text{blanket})=P(X\mid\text{everything else})$.
4. **Implement** d-separation (undirected paths, active/inactive triples, the collider-descendant rule) and **reproduce** all 14 worked examples of slides 64–93 plus Example 15.
5. **Verify** d-separation by brute force on 672 queries with random CPTs, and **explain** the difference between *guaranteed independence* and *dependence* (faithfulness), with a biological counterexample.
6. **Test** two d-separation predictions of the Sachs consensus network on real single-cell data with a permutation test, and **interpret** a failed prediction honestly.
7. **Connect** the rules to biology: co-regulated genes, collider bias in case-only studies, what d-separation can and cannot tell a biologist.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, exact-inference helpers | 5 min |
| 1 · Independence questions | definitions; the Markov chain questions of slide 45 | 15 min |
| 2 · Three canonical triples | chain, common cause, common effect; collider bias; explaining away | 25 min |
| 3 · Markov blankets | parents + children + co-parents; alarm, TF network, Sachs network | 20 min |
| 4 · d-separation: the algorithm | paths, triples, the collider rule; `d_separated` | 20 min |
| 5 · The 14 worked examples | slides 64–93 reproduced, drawn and explained | 25 min |
| 6 · Brute-force verification | random CPTs, 672 queries; faithfulness and its failure | 15 min |
| 7 · Example 15, biology and real data | co-regulated genes; testing the Sachs network on real cells | 20 min |
| ✅ / 🧠 / 🚀 | summary, checkpoint, discussion | 5 min |

## 🧫 The biological question

Across 500 breast tumours, the expression levels of two genes rise and fall together. Does one gene regulate the
other? Or are both switched on by the same transcription factor — in which case the correlation should **vanish**
once you account for that factor's activity? And if you only study *patients* (people who already have a disease),
two genetic risk factors that are independent in the population can suddenly look **negatively** associated. Which
correlations should a biologist expect, and which are artefacts of what was measured or who was sampled?

A Bayesian network answers these questions **from the graph alone**, before any data are collected:

- **Markov blanket:** which proteins must I measure to predict Raf's activity as well as possible? (Its parents, its
  children, and its children's other parents.)
- **d-separation:** are Mek and Akt independent once Erk is known? Are p38 and JNK independent once PKA and PKC
  are known? Each such statement is a *testable prediction* of the network.

This notebook follows slides 38–96 of the 02-620 lecture: it builds the rules, reproduces every worked example,
verifies each claim by brute-force computation — and finally tests two predictions of the Sachs *et al.* (2005)
signalling network on the real single-cell measurements. One of them fails, and we will discuss why.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
They provide the alarm network of the slides, the Sachs et al. 2005 consensus network and data loader (with a simulated fallback when offline), and the small graph-drawing function. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_DATA_URL = "https://raw.githubusercontent.com/xulabs/edu/main/ml/tutorial/BiologyToML/datasets/{name}"


CACHE_DIR = os.environ.get("BIO2ML_CACHE", "course_data")


def fetch_course_file(name: str, timeout: float = 60.0) -> str | None:
    """Return a local path to the course data file ``name``, or ``None`` if it cannot be found.

    Looks in the repository's ``datasets/`` folder first (when the notebook runs inside a
    clone of xulabs/edu), then in the download cache, and finally downloads it from GitHub.
    """
    for base in ("../datasets", "datasets", "ml/tutorial/BiologyToML/datasets", CACHE_DIR):
        path = os.path.join(base, name)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
    dest = os.path.join(CACHE_DIR, name)
    os.makedirs(CACHE_DIR, exist_ok=True)
    try:
        with urllib.request.urlopen(REPO_DATA_URL.format(name=name), timeout=timeout) as r, \
                open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return dest
    except Exception as e:  # noqa: BLE001 — any network problem → offline fallback
        print(f"⚠️  Could not obtain {name} ({type(e).__name__}); using a simulated stand-in instead.")
        return None


ALARM_PARENTS = {"B": (), "E": (), "A": ("B", "E"), "J": ("A",), "M": ("A",)}


ALARM_CPTS = {
    "B": {(): 0.05},
    "E": {(): 0.10},
    "A": {(1, 1): 0.95, (1, 0): 0.85, (0, 1): 0.50, (0, 0): 0.05},
    "J": {(1,): 0.70, (0,): 0.05},
    "M": {(1,): 0.80, (0,): 0.15},
}


SACHS_PROTEINS = ["raf", "mek", "plc", "pip2", "pip3", "erk", "akt", "pka", "pkc", "p38", "jnk"]


SACHS_CONSENSUS_EDGES = [
    ("pka", "raf"), ("pkc", "raf"), ("raf", "mek"), ("pka", "mek"), ("pkc", "mek"),
    ("pip3", "plc"), ("plc", "pip2"), ("pip3", "pip2"), ("mek", "erk"), ("pka", "erk"),
    ("pip3", "akt"), ("erk", "akt"), ("pka", "akt"), ("pkc", "pka"), ("plc", "pkc"),
    ("pip2", "pkc"), ("pka", "p38"), ("pkc", "p38"), ("pka", "jnk"), ("pkc", "jnk"),
]


def simulate_sachs_like(rng: np.random.Generator, n: int = 2603, kind: str = "continuous") -> pd.DataFrame:
    """Simulated stand-in for the Sachs data: a linear-Gaussian model on the consensus network."""
    order = ["pip3", "plc", "pip2", "pkc", "pka", "raf", "mek", "erk", "akt", "p38", "jnk"]  # topological
    parents = {p: [a for a, b in SACHS_CONSENSUS_EDGES if b == p] for p in SACHS_PROTEINS}
    data = {}
    for node in order:
        value = rng.normal(0, 1, n)
        for par in parents[node]:
            value = value + rng.choice([-1, 1]) * rng.uniform(0.4, 0.8) * data[par]
        data[node] = (value - value.mean()) / value.std()
    df = pd.DataFrame({p: data[p] for p in SACHS_PROTEINS})
    if kind == "discrete":
        df = df.apply(lambda c: pd.qcut(c, 3, labels=False)).astype(int)
    return df


def load_sachs(kind: str = "continuous", observational_only: bool = True) -> dict:
    """Sachs et al. (2005) single-cell phospho-protein measurements (flow cytometry).

    ``kind='continuous'`` gives log-scale intensities (7,466 cells), ``'discrete'`` the
    low/medium/high (0/1/2) version (5,400 cells).  With ``observational_only=True`` only
    cells without a targeted perturbation are returned.  Returns a dict with ``data``
    (DataFrame, 11 proteins), ``intervention``, ``edges`` (consensus network) and ``source``.
    """
    path = fetch_course_file(f"sachs_cytometry_{kind}.csv.gz")
    if path is None:
        df = simulate_sachs_like(np.random.default_rng(2005), kind=kind)
        ivn, source = np.array(["none"] * len(df)), "SIMULATED stand-in (offline)"
    else:
        full = pd.read_csv(path)
        df, ivn, source = full[SACHS_PROTEINS], full["intervention"].to_numpy(), "Sachs et al. 2005 (real data)"
    if observational_only:
        keep = ivn == "none"
        df, ivn = df[keep].reset_index(drop=True), ivn[keep]
    return dict(data=df, intervention=ivn, edges=list(SACHS_CONSENSUS_EDGES), source=source)


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['fetch_course_file', 'simulate_sachs_like', 'load_sachs', 'draw_graph']))

### 🛠️ Exact-inference helpers (from Module 15)

To check every independence claim we need exact probabilities. For networks of up to ~15 binary variables we can
simply list all $2^n$ configurations and their probabilities (Module 15's `full_joint`, here vectorised with NumPy),
then answer any question by adding up rows. The class below stores that table and answers $P(\text{event}\mid
\text{given})$; `marginal` returns the joint distribution of a few chosen variables as an array.

In [ ]:
# ── 0.1 A joint table for any small binary Bayesian network ──
ALARM_POS = {"B": (0.2, 0.85), "E": (0.8, 0.85), "A": (0.5, 0.5), "J": (0.25, 0.15), "M": (0.75, 0.15)}


def edges_from_parents(parents):
    """Convert a parents dictionary into a list of (parent, child) edges."""
    return [(p, child) for child, pa in parents.items() for p in pa]


class JointTable:
    """All 2^n states of a binary BN (ALARM-style CPTs: P(node = 1 | parent values)) and their probabilities."""

    def __init__(self, parents, cpts):
        self.nodes = list(parents)
        self.col = {v: i for i, v in enumerate(self.nodes)}
        self.states = np.array(list(itertools.product([0, 1], repeat=len(self.nodes))))
        self.p = np.ones(len(self.states))
        for node, pa in parents.items():                        # multiply in one factor per node
            for pa_vals, theta in cpts[node].items():
                rows = np.all(self.states[:, [self.col[q] for q in pa]] == pa_vals, axis=1) if pa \
                    else np.ones(len(self.states), bool)
                x = self.states[rows, self.col[node]]
                self.p[rows] *= np.where(x == 1, theta, 1 - theta)

    def _rows(self, cond):
        mask = np.ones(len(self.states), bool)
        for var, val in cond.items():
            mask &= self.states[:, self.col[var]] == val
        return mask

    def prob(self, event, given=None):
        """P(event | given); event and given are dicts {variable: 0/1}."""
        given = given or {}
        return self.p[self._rows({**event, **given})].sum() / self.p[self._rows(given)].sum()

    def marginal(self, variables):
        """Joint distribution of `variables` as an array with one axis (of length 2) per variable."""
        code = self.states[:, [self.col[v] for v in variables]] @ (2 ** np.arange(len(variables))[::-1])
        return np.bincount(code, weights=self.p, minlength=2 ** len(variables)).reshape([2] * len(variables))


alarm = JointTable(ALARM_PARENTS, ALARM_CPTS)
print(f"{len(alarm.p)} states, total probability {alarm.p.sum():.6f}, "
      f"P(B,¬E,A,J,¬M) = {alarm.prob({'B': 1, 'E': 0, 'A': 1, 'J': 1, 'M': 0}):.6f}")

---
## 1 · Independence questions

> 📍 **Where we are.** Module 15 showed that the *missing* arrows of a Bayesian network save parameters. Each missing
> arrow is an independence assumption. This section makes "independence" precise and gives two numerical tests we
> will use throughout (slides 39–49).

**Two ways to say "independent" (slides 42–43).** $A$ and $B$ are independent, written $A\perp B$, if and only if

$$P(A,B)=P(A)\,P(B) \qquad\Longleftrightarrow\qquad P(A\mid B)=P(A).$$

*In words:* the joint probability is the product of the separate probabilities; equivalently, learning $B$ does not
change what you believe about $A$. (Why equivalent? If $P(A\mid B)=P(A)$ then $P(A,B)=P(A\mid B)P(B)=P(A)P(B)$.)

**Conditional independence (slide 44).** $A$ and $B$ are independent given $C$, written $A\perp B\mid C$, iff

$$P(A,B\mid C)=P(A\mid C)\,P(B\mid C) \qquad\Longleftrightarrow\qquad P(A\mid B,C)=P(A\mid C).$$

*In words:* once you know $C$, learning $B$ tells you nothing more about $A$ — for **every** value of $C$.

🧮 **Math refresher — measuring dependence with one number.** Checking "$P(A\mid B,C)=P(A\mid C)$ for all values" gives
many numbers; we report the largest violation, $\max|P(a\mid b,c)-P(a\mid c)|$. A single summary is the
**conditional mutual information** (in bits; $\log_2$ as everywhere in the course):

$$I(A;B\mid C)=\sum_{a,b,c}P(a,b,c)\,\log_2\frac{P(a,b\mid c)}{P(a\mid c)\,P(b\mid c)} .$$

*In words:* the average, over all configurations, of how much the true joint (given $c$) differs from the
"independent" product, measured on a log scale. It is **never negative**, and it is **exactly zero if and only if**
$A\perp B\mid C$. With $C$ empty it is the ordinary mutual information $I(A;B)$, which reappears in Module 17
(Chow–Liu trees). $A$, $B$ and $C$ may be *sets* of variables.

In [ ]:
# ── 1.1 Two numerical tests of conditional independence ──
def cond_mutual_info(table, X, Y, Z=()):
    """I(X;Y|Z) in bits for lists of variables X, Y, Z of a JointTable (0 ⇔ conditionally independent)."""
    X, Y, Z = list(X), list(Y), list(Z)
    p = table.marginal(X + Y + Z).reshape(2 ** len(X), 2 ** len(Y), 2 ** len(Z))
    pz, pxz, pyz = p.sum(axis=(0, 1)), p.sum(axis=1), p.sum(axis=0)
    ratio = p * pz[None, None, :] / (pxz[:, None, :] * pyz[None, :, :] + 1e-300)
    keep = p > 0
    return float(np.sum(p[keep] * np.log2(ratio[keep])))


def max_cond_diff(table, X, Y, Z=()):
    """max over values of |P(x | y, z) − P(x | z)| (0 ⇔ conditionally independent)."""
    X, Y, Z = list(X), list(Y), list(Z)
    p = table.marginal(X + Y + Z).reshape(2 ** len(X), 2 ** len(Y), 2 ** len(Z))
    p_x_given_yz = p / np.maximum(p.sum(axis=0, keepdims=True), 1e-300)
    p_x_given_z = p.sum(axis=1, keepdims=True) / p.sum(axis=(0, 1), keepdims=True)
    return float(np.max(np.abs(p_x_given_yz - p_x_given_z)[:, p.sum(axis=0) > 0]))


print(f"B, E  : I(B;E) = {cond_mutual_info(alarm, ['B'], ['E']):.2e} bits   max diff = {max_cond_diff(alarm, ['B'], ['E']):.2e}")
print(f"B, J  : I(B;J) = {cond_mutual_info(alarm, ['B'], ['J']):.2e} bits   max diff = {max_cond_diff(alarm, ['B'], ['J']):.2e}")
print(f"B, J|A: I(B;J|A) = {cond_mutual_info(alarm, ['B'], ['J'], ['A']):.2e} bits   max diff = {max_cond_diff(alarm, ['B'], ['J'], ['A']):.2e}")

🔍 **What just happened?** Both tests agree. Burglary and earthquake are independent ($I=0$ up to round-off): the
network has no arrow between them and no common cause. Burglary and John's call are **dependent**
($I(B;J)>0$): a burglary makes the alarm, and hence the call, more likely. But **given the alarm**, John's call says
nothing more about a burglary: $I(B;J\mid A)=0$. These are exactly the two statements of slide 41: the Markov blanket
statement $B\perp J,M\mid A,E$ and the d-separation statement $B\perp J\mid A$.

### 1.1 The Markov chain questions (slide 45)

The simplest network is a chain $X_1\to X_2\to X_3\to X_4$ — think of a signalling cascade: receptor → kinase 1 →
kinase 2 → transcription factor. Its factorization is

$$P(X_1,X_2,X_3,X_4)=P(X_1)\,P(X_2\mid X_1)\,P(X_3\mid X_2)\,P(X_4\mid X_3).$$

*In words:* each step depends only on the step immediately before it. Slide 45 asks three questions. Predict the
answers, then run the cell (the CPT values are illustrative).

In [ ]:
# ── 1.2 A four-step signalling cascade: X1 → X2 → X3 → X4 ──
CHAIN4_PARENTS = {"X1": (), "X2": ("X1",), "X3": ("X2",), "X4": ("X3",)}
step = {(1,): 0.9, (0,): 0.2}                     # P(next step active | previous step active / inactive)
CHAIN4_CPTS = {"X1": {(): 0.5}, "X2": step, "X3": step, "X4": step}
chain4 = JointTable(CHAIN4_PARENTS, CHAIN4_CPTS)

print(f"P(X4=1) = {chain4.prob({'X4': 1}):.4f};  P(X4=1 | X2=1) = {chain4.prob({'X4': 1}, {'X2': 1}):.4f};  "
      f"P(X4=1 | X2=0) = {chain4.prob({'X4': 1}, {'X2': 0}):.4f}")
for x3 in (0, 1):
    vals = [chain4.prob({"X4": 1}, {"X2": x2, "X3": x3}) for x2 in (0, 1)]
    print(f"P(X4=1 | X2=0, X3={x3}) = {vals[0]:.4f}   P(X4=1 | X2=1, X3={x3}) = {vals[1]:.4f}")
for question, X, Y, Z in [("Is X2 ⊥ X4?", ["X2"], ["X4"], []), ("Is X2 ⊥ X4 | X3?", ["X2"], ["X4"], ["X3"]),
                          ("Is X1 ⊥ X4 | X2?", ["X1"], ["X4"], ["X2"])]:
    cmi = cond_mutual_info(chain4, X, Y, Z)
    print(f"{question:18s} I = {cmi:.2e} bits → {'yes' if cmi < 1e-12 else 'no'}")

🔍 **What just happened?** The answers match slides 47–49: (1) **No**, $X_2$ and $X_4$ are dependent —
$P(X_4{=}1\mid X_2)$ changes with $X_2$. (2) **Yes**, $X_2\perp X_4\mid X_3$: once the middle step is known, the earlier
step adds nothing ($P(X_4{=}1\mid X_2,X_3)$ depends only on $X_3$). (3) **Yes**, $X_1\perp X_4\mid X_2$.

Why (2) holds for *any* CPTs: sum out $X_1$ from the factorization,

$$P(x_2,x_3,x_4)=\Big[\sum_{x_1}P(x_1)P(x_2\mid x_1)\Big]P(x_3\mid x_2)P(x_4\mid x_3)=P(x_2)\,P(x_3\mid x_2)\,P(x_4\mid x_3),$$

so $P(x_4\mid x_2,x_3)=P(x_2,x_3,x_4)/P(x_2,x_3)=P(x_4\mid x_3)$.

*In words:* in a chain, the present screens off the past from the future — the **Markov property** that hidden Markov
models (Modules 19–20) are built on.

✏️ **Your turn.** Make the cascade "leakier": change `step` to `{(1,): 0.6, (0,): 0.4}`. Do the three answers change?
Does the *size* of the dependence $I(X_2;X_4)$ change? (Independence is a property of the graph; the strength of a
dependence depends on the CPTs.)

### 📑 Slide connection — `008_BN_mz_021825`, slides 38–49

- **What the slides show.** Overview (slides 38–39): inference "on the graph" (Markov blanket, d-separation) before inference on the
    distribution. Given a BN one can answer conditional-independence questions "simply by examining the graph
    structure" (slide 40); slide 41 previews $B\perp J,M\mid A,E$ (Markov blanket) and $B\perp J\mid A$ (d-separation).
    Definitions of independence and conditional independence (slides 42–44). The Markov chain $X_1\to X_2\to X_3\to X_4$
    with the answers No, Yes, Yes (slides 45–49).
- **What this notebook adds.** A vectorised exact joint table for any small binary BN; two numerical tests of conditional independence
    (largest violation and conditional mutual information, defined here and reused in Module 17); the algebra showing
    why the chain answers hold for every choice of CPTs.
- **Exam tip.** Independence must hold for *all* values: $P(A\mid B,C)=P(A\mid C)$ for every $a,b,c$. One matching pair of
    numbers proves nothing. And "independent" and "conditionally independent" are different claims — neither
    implies the other (Section 2).

---
## 2 · The three canonical triples

> 📍 **Where we are.** Every path through a Bayesian network is made of overlapping *triples* of consecutive nodes,
> and there are only three kinds. If you understand these three, you understand d-separation (Section 4, slide 62).

| Triple | Graph | Factorization | Without evidence | Observing the middle node $B$ |
|---|---|---|---|---|
| **causal chain** | $X\to B\to Y$ | $P(x)P(b\mid x)P(y\mid b)$ | dependent | **blocks**: $X\perp Y\mid B$ |
| **common cause** | $X\leftarrow B\to Y$ | $P(b)P(x\mid b)P(y\mid b)$ | dependent | **blocks**: $X\perp Y\mid B$ |
| **common effect** (collider, v-structure) | $X\to B\leftarrow Y$ | $P(x)P(y)P(b\mid x,y)$ | **independent** | **opens**: dependent given $B$ |

The collider row is the surprising one. Summing out $B$ from its factorization gives

$$P(x,y)=\sum_b P(x)\,P(y)\,P(b\mid x,y)=P(x)\,P(y)\sum_b P(b\mid x,y)=P(x)\,P(y),$$

*In words:* two independent causes stay independent as long as you do not look at their common effect. But given the
effect, they compete to explain it: if one cause is present, the other is needed less. Observing any **descendant** of
the collider (an effect of the effect) also opens it, because the descendant carries information about $B$.

In [ ]:
# ── 2.1 The three triples (plus a collider with a descendant C), checked exactly ──
TRIPLES = {    # illustrative CPTs; read X, B, Y as gene X, transcription factor B, target Y, etc.
    "causal chain X→B→Y": ({"X": (), "B": ("X",), "Y": ("B",)},
                           {"X": {(): 0.5}, "B": {(1,): 0.8, (0,): 0.1}, "Y": {(1,): 0.9, (0,): 0.2}}),
    "common cause X←B→Y": ({"B": (), "X": ("B",), "Y": ("B",)},
                           {"B": {(): 0.4}, "X": {(1,): 0.8, (0,): 0.1}, "Y": {(1,): 0.9, (0,): 0.2}}),
    "common effect X→B←Y (B→C)": ({"X": (), "Y": (), "B": ("X", "Y"), "C": ("B",)},
                                  {"X": {(): 0.3}, "Y": {(): 0.4},
                                   "B": {(0, 0): 0.05, (0, 1): 0.6, (1, 0): 0.7, (1, 1): 0.9},
                                   "C": {(1,): 0.8, (0,): 0.1}}),
}
rows = []
for name, (parents, cpts) in TRIPLES.items():
    tab = JointTable(parents, cpts)
    row = {"triple": name, "I(X;Y)": cond_mutual_info(tab, ["X"], ["Y"]),
           "I(X;Y | B)": cond_mutual_info(tab, ["X"], ["Y"], ["B"])}
    if "C" in parents:
        row["I(X;Y | C)"] = cond_mutual_info(tab, ["X"], ["Y"], ["C"])
    rows.append(row)
pd.set_option("display.float_format", lambda v: f"{v:.2e}")
print(pd.DataFrame(rows).set_index("triple").to_string(na_rep="—"))
pd.reset_option("display.float_format")

🔍 **What just happened?** Read the table row by row. In the chain and the common-cause triple, $X$ and $Y$ share
information ($I>0$) until $B$ is observed, then $I$ drops to zero (round-off level). The collider does the opposite:
$I(X;Y)$ is exactly zero, and observing $B$ — or just its descendant $C$ — creates dependence. Observing the
descendant opens the collider *less* than observing $B$ itself, because $C$ is only a noisy report of $B$.

### 2.1 Collider bias in a case-only study (simulation)

Colliders are not a curiosity. Suppose two genetic risk variants $G_1$ and $G_2$ (carrier yes/no) are inherited
independently and each raises the risk of a disease $D$. In the population, $G_1\to D\leftarrow G_2$ is a collider.
A study that recruits **only patients** conditions on $D=1$ — and opens the collider. We simulate 200,000 people.

In [ ]:
# ── 2.2 SIMULATED case-only study: two independent risk variants become negatively associated ──
rng = np.random.default_rng(SEED + 2)
n_people = 200_000
g1 = (rng.random(n_people) < 0.2).astype(int)                 # carrier of risk variant 1 (20 % of people)
g2 = (rng.random(n_people) < 0.3).astype(int)                 # carrier of risk variant 2 (30 %), independent of g1
p_disease = 1 - (1 - 0.02) * (1 - 0.25 * g1) * (1 - 0.35 * g2)   # each variant adds risk ("noisy-OR")
d = (rng.random(n_people) < p_disease).astype(int)

def carrier2_rate(mask):
    """Fraction of G2 carriers among the people selected by `mask`."""
    return g2[mask].mean()

print(f"everyone ({n_people:,}): P(G2 | G1=1) = {carrier2_rate(g1 == 1):.3f}   P(G2 | G1=0) = {carrier2_rate(g1 == 0):.3f}")
print(f"cases only ({d.sum():,}):  P(G2 | G1=1, D=1) = {carrier2_rate((g1 == 1) & (d == 1)):.3f}   "
      f"P(G2 | G1=0, D=1) = {carrier2_rate((g1 == 0) & (d == 1)):.3f}")

fig, ax = plt.subplots(figsize=(5.5, 3))
labels = ["everyone", "cases only (D = 1)"]
x = np.arange(2)
ax.bar(x - 0.18, [carrier2_rate(g1 == 1), carrier2_rate((g1 == 1) & (d == 1))], 0.36, label="G1 carriers", color="tab:blue")
ax.bar(x + 0.18, [carrier2_rate(g1 == 0), carrier2_rate((g1 == 0) & (d == 1))], 0.36, label="G1 non-carriers", color="tab:orange")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("fraction carrying G2")
ax.set_title("SIMULATED: conditioning on the disease (a collider)\ncreates a spurious negative association")
ax.legend(); plt.tight_layout(); plt.show()

🔍 **What just happened?** In the whole population the G2 carrier rate is about 0.30 whether or not someone carries
G1 — independence, as simulated. Among patients the picture flips: patients *without* G1 mostly carry G2 (their
disease needs an explanation), patients *with* G1 much less often. A naïve analysis of the case-only data would
"discover" that the two variants protect against each other. Epidemiologists call this **collider bias** (or
Berkson's paradox); in a graph it is simply "observing a common effect opens the path".

### 2.2 Explaining away in the alarm network

The alarm $A$ is a collider between burglary $B$ and earthquake $E$. A classic exam exercise: compute
$P(B{=}1\mid A{=}1)$ and $P(B{=}1\mid A{=}1,E{=}1)$ exactly.

In [ ]:
# ── 2.3 Explaining away: an earthquake "explains" the alarm and lowers the probability of a burglary ──
queries = [("P(B=1)", {}), ("P(B=1 | E=1)", {"E": 1}), ("P(B=1 | A=1)", {"A": 1}),
           ("P(B=1 | A=1, E=0)", {"A": 1, "E": 0}), ("P(B=1 | A=1, E=1)", {"A": 1, "E": 1}),
           ("P(B=1 | J=1)", {"J": 1}), ("P(B=1 | J=1, E=1)", {"J": 1, "E": 1})]
values = [alarm.prob({"B": 1}, given) for _, given in queries]
for (label, _), v in zip(queries, values):
    print(f"{label:20s} = {v:.4f}")

fig, ax = plt.subplots(figsize=(7, 3))
colors = ["0.6", "0.6", "tab:blue", "tab:green", "tab:red", "tab:blue", "tab:red"]
ax.barh([q for q, _ in queries][::-1], values[::-1], color=colors[::-1])
ax.set_xlabel("probability of a burglary"); ax.set_title("explaining away in the alarm network (slide CPTs)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Before any evidence, $P(B{=}1)=0.05$, and learning about an earthquake alone changes nothing
($B\perp E$). Hearing the alarm raises the burglary probability to about 0.32. Now add the earthquake: the alarm is
"explained", and the burglary probability drops to about 0.09. If instead we learn that there was **no** earthquake,
the burglary probability rises to about 0.47 — something must have set off the alarm. John's call (a *descendant* of
the collider) behaves like a weaker version of the alarm: it raises $P(B)$, and an earthquake again explains part of
it away. **Given the alarm, $B$ and $E$ are dependent** although they are independent a priori.

✏️ **Your turn.** Compute $P(E{=}1\mid A{=}1)$ and $P(E{=}1\mid A{=}1,B{=}1)$. Does a burglary explain away the
earthquake by as much? Why is the effect asymmetric? (Look at the CPT of A.)

---
## 3 · Markov blankets

> 📍 **Where we are.** We now ask a question about *one* node against *all* the others: what is the smallest set of
> variables that tells us everything the rest of the network could tell us about it (slides 50–59)?

Take a node, say $B$, and the set $S$ of all other nodes. We look for a subset $S_1\subset S$ such that
$B\perp S\setminus S_1\mid S_1$ ("$S\setminus S_1$" means the nodes of $S$ that are not in $S_1$). Such an $S_1$ contains all
the information about $B$. The answer is the **(minimal) Markov blanket**:

> **Markov blanket of $X$ = its parents + its children + the other parents of its children (co-parents).**

**Why exactly these nodes?** In the conditional distribution of $X$ given everything else, every factor of the joint
distribution that does not contain $X$ cancels between numerator and denominator:

$$P(x\mid \text{all others})=\frac{P(x\mid \mathrm{pa}(x))\prod_{c\in\mathrm{ch}(x)}P(c\mid \mathrm{pa}(c))}{\sum_{x'}P(x'\mid \mathrm{pa}(x))\prod_{c\in\mathrm{ch}(x)}P(c\mid \mathrm{pa}(c))\big|_{x\to x'}} .$$

*In words:* only $X$'s own factor (which involves its parents) and its children's factors (which involve the children
and *their* parents, i.e. the co-parents) survive. Nothing else in the network matters once these are known. The
co-parents are needed because of explaining away: a child is a collider between $X$ and its other parents.

In [ ]:
# ── 3.1 Markov blankets in the alarm network (slides 52–57) ──
def markov_blanket(parents, node):
    """Parents, children and co-parents (other parents of the children) of `node`."""
    children = [c for c, pa in parents.items() if node in pa]
    coparents = {p for c in children for p in parents[c] if p != node}
    return sorted(set(parents[node]) | set(children) | coparents)


rows = []
for node in ALARM_PARENTS:
    mb = markov_blanket(ALARM_PARENTS, node)
    rest = [v for v in ALARM_PARENTS if v != node and v not in mb]
    rows.append({"node": node, "Markov blanket": ", ".join(mb), "rest": ", ".join(rest) or "—",
                 "I(node; rest | blanket)": cond_mutual_info(alarm, [node], rest, mb) if rest else 0.0})
print(pd.DataFrame(rows).to_string(index=False))

# dropping the co-parent E from B's blanket breaks the property
print(f"\nwith only A as 'blanket' of B:  I(B; E,J,M | A) = {cond_mutual_info(alarm, ['B'], ['E', 'J', 'M'], ['A']):.4f} bits")
print(f"P(B=1 | E,A,J,M) = P(B=1 | E,A)?  max |diff| = {max_cond_diff(alarm, ['B'], ['J', 'M'], ['E', 'A']):.1e}")

🔍 **What just happened?** The function reproduces the slides: the blanket of B is {A, E} (child A and co-parent E),
so $P(B\mid E,A,J,M)=P(B\mid E,A)$ (slide 53); the blanket of A is everything else, {B, E, J, M} (slide 55); the
blanket of J is just {A}, so $P(J\mid B,E,A,M)=P(J\mid A)$ (slide 57). For every node the conditional mutual information
between the node and the rest of the network, given the blanket, is zero. If we forget the co-parent E and condition
on A alone, B is no longer independent of the rest ($I>0$): that is explaining away again.

### 3.1 Markov blankets in a gene regulatory network (slides 58–59)

Slides 58–59 draw three layers: transcription factors (TFs), the genes encoding further TFs, and their target genes.
Below is a small network of the same shape (regulators R, TF genes G, targets T). Which nodes form the Markov blanket of
TF gene G2? Predict first.

In [ ]:
# ── 3.2 The Markov blanket of a TF gene (a network shaped like slides 58–59) ──
GRN_EDGES = [("R1", "G1"), ("R1", "G2"), ("R2", "G2"), ("R2", "G3"), ("R3", "G4"),
             ("G1", "T1"), ("G1", "T2"), ("G2", "T2"), ("G2", "T3"), ("G3", "T3"), ("G3", "T4"),
             ("G4", "T4"), ("G4", "T5")]
GRN_POS = {"R1": (0.2, 0.9), "R2": (0.5, 0.9), "R3": (0.8, 0.9),
           "G1": (0.1, 0.5), "G2": (0.37, 0.5), "G3": (0.63, 0.5), "G4": (0.9, 0.5),
           "T1": (0.03, 0.1), "T2": (0.27, 0.1), "T3": (0.5, 0.1), "T4": (0.73, 0.1), "T5": (0.97, 0.1)}
GRN_NODES = list(GRN_POS)
grn_parents = {v: tuple(u for u, w in GRN_EDGES if w == v) for v in GRN_NODES}


def mark_nodes(ax, pos, nodes, color="tab:red", half=0.085):
    """Draw dashed squares around `nodes` (the slides' way of marking query nodes)."""
    for node in nodes:
        x, y = pos[node]
        ax.add_patch(Rectangle((x - half, y - half), 2 * half, 2 * half, fill=False, ls="--", ec=color, lw=1.3))


mb_g2 = markov_blanket(grn_parents, "G2")
print("Markov blanket of G2:", mb_g2)
grn_cpts = {v: {vals: p for vals, p in zip(itertools.product([0, 1], repeat=len(pa)),
                                             np.random.default_rng(SEED + 3 + i).uniform(0.05, 0.95, 2 ** len(pa)))}
            for i, (v, pa) in enumerate(grn_parents.items())}          # random CPTs, 2^12 = 4,096 states
grn = JointTable(grn_parents, grn_cpts)
rest = [v for v in GRN_NODES if v != "G2" and v not in mb_g2]
print(f"I(G2; {', '.join(rest)} | blanket) = {cond_mutual_info(grn, ['G2'], rest, mb_g2):.1e} bits (random CPTs)")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
draw_graph(GRN_EDGES, GRN_POS, ax=ax, highlight=mb_g2, radius=0.055,
           title="Markov blanket (orange) of TF gene G2 (dashed square)")
mark_nodes(ax, GRN_POS, ["G2"])
for y, label in [(0.9, "regulators"), (0.5, "TF genes"), (0.1, "targets")]:
    ax.text(-0.09, y, label, ha="right", va="center", fontsize=8, color="0.3")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The blanket of G2 is its two regulators R1 and R2 (parents), its targets T2 and T3
(children), and G1 and G3 — the *other* TF genes that co-regulate those targets (co-parents). With random CPTs the
remaining five nodes carry no further information about G2 ($I\approx10^{-16}$ bits). Biologically: to predict a TF
gene's activity you need its regulators, its targets, and the other factors acting on those targets. Note *why* G1 and
G3 are in the blanket: not because they share a regulator with G2 (that alone would not matter once R1 and R2 are
known), but because they co-regulate G2's targets. G4, the third regulator R3 and the targets T1, T4, T5 are
irrelevant once the blanket is known.

Now the real curated network of Module 15.

In [ ]:
# ── 3.3 Markov blankets in the Sachs consensus signalling network ──
sachs_parents = {p: tuple(a for a, b in SACHS_CONSENSUS_EDGES if b == p) for p in SACHS_PROTEINS}
for protein in SACHS_PROTEINS:
    mb = markov_blanket(sachs_parents, protein)
    print(f"{protein:5s} blanket ({len(mb):2d}): {', '.join(mb)}")

🔍 **What just happened?** In the consensus network, Raf's blanket is {Mek, PKA, PKC}: its parents PKA and PKC, its
child Mek, and Mek's other parents (again PKA and PKC). If the consensus network were exactly right, these three
proteins would predict Raf's phosphorylation as well as all ten others together. PKA and PKC, the "hubs" of the
network, have blankets containing nearly every other protein. A Markov blanket is also a principled answer to *feature
selection*: the best predictor set for a variable is its blanket.

✏️ **Your turn.** (a) Which protein has the smallest blanket? Is it a root, a leaf, or neither? (b) Add an edge
`("erk", "raf")` (the known Erk → Raf feedback) to `SACHS_CONSENSUS_EDGES` in a copy. What happens to Raf's
blanket — and is the graph still a DAG? (Use the topological sort of Module 15, or reason it out.)

### 📑 Slide connection — `008_BN_mz_021825`, slides 50–59

- **What the slides show.** The Markov blanket defines conditional independence between one variable and the rest: find $S_1\subset S$
    with $B\perp S\setminus S_1\mid S_1$; in a BN a variable is conditionally independent of all others given its
    (minimal) Markov blanket = parents, children and co-parents of children. Examples: blanket of B = {E, A}
    ($P(B\mid E,A,J,M)=P(B\mid E,A)$), of A = {B, E, J, M}, of J = {A} ($P(J\mid B,E,A,M)=P(J\mid A)$). Slides 58–59: the
    blanket of a node in a TF → TF-gene → target-gene network.
- **What this notebook adds.** A derivation of *why* the blanket suffices (all other factors cancel); `markov_blanket` for any DAG; a
    numerical check (conditional mutual information exactly 0) for every alarm node and a 12-node regulatory network
    with random CPTs; the counter-check that dropping a co-parent breaks the property; blankets of the 11 Sachs
    proteins.
- **Exam tip.** Students forget the **co-parents**. Mnemonic: everything that shares a factor with $X$ in the
    factorization — its own factor $P(X\mid\mathrm{pa})$ and its children's factors $P(c\mid \mathrm{pa}(c))$.

---
## 4 · d-separation: the algorithm

> 📍 **Where we are.** The Markov blanket answers "one node versus all the rest". d-separation answers the general
> question for **two** nodes and **any** set of evidence (slides 61–63). It combines the three triples of Section 2
> along every path of the graph.

**The question (slide 61).** Are $X$ and $Y$ conditionally independent given the evidence variables in a set
$\{Z\}$? ⚠️ The slides write the evidence set as $\{Z\}$ although one of the nodes in the examples is also called
$Z$; in this notebook the evidence set is called **evidence** (code: `evidence`).

**The rules (slide 62).**
- A **path** is a sequence of distinct nodes, each joined to the next by an edge, **ignoring the arrow directions**.
- A triple $a - b - c$ on a path is **inactive** if
  - it is a causal chain ($a\to b\to c$ or $a\leftarrow b\leftarrow c$) and $b$ is observed, or
  - it is a common cause ($a\leftarrow b\to c$) and $b$ is observed, or
  - it is a common effect ($a\to b\leftarrow c$) and **neither $b$ nor any of its descendants** is observed.
- One inactive triple **blocks** the whole path.

**The algorithm (slide 63).** List all (undirected) paths between $X$ and $Y$; check each for an inactive triple. If
**all paths are blocked**, $X$ and $Y$ are **d-separated** by the evidence, and then it is *guaranteed* that
$X\perp Y\mid\text{evidence}$ in every distribution that factorizes over the graph. If at least one path is active, the
graph does not guarantee independence ("independence not guaranteed").

🧮 **Paths and depth-first search.** We find all paths by extending partial paths one neighbour at a time and never
revisiting a node (a depth-first search, like `descendants_of` in Module 15). The number of paths can grow
exponentially in large graphs; practical software uses an equivalent linear-time "reachability" algorithm
(Koller & Friedman 2009, §3.3.3; the closely related "Bayes-ball" formulation is due to Shachter 1998), but path
enumeration is exactly what the slides teach.

In [ ]:
# ── 4.1 Paths through a graph, ignoring arrow directions ──
def children_of(edges, node):
    """Nodes that `node` points to."""
    return sorted({v for u, v in edges if u == node})


def descendants_of(edges, node):
    """All nodes reachable from `node` by following arrows."""
    found, stack = set(), [node]
    while stack:
        for child in children_of(edges, stack.pop()):
            if child not in found:
                found.add(child)
                stack.append(child)
    return found


def undirected_paths(edges, x, y):
    """All paths from x to y that never revisit a node, ignoring arrow directions."""
    nbrs = {}
    for u, v in edges:
        nbrs.setdefault(u, set()).add(v)
        nbrs.setdefault(v, set()).add(u)
    paths, stack = [], [[x]]
    while stack:
        path = stack.pop()
        for nxt in sorted(nbrs.get(path[-1], ())):
            if nxt == y:
                paths.append(path + [y])
            elif nxt not in path:
                stack.append(path + [nxt])
    return sorted(paths, key=lambda p: (len(p), p))

In [ ]:
# ── 4.2 Triples and the d-separation test, exactly as on slides 62–63 ──
def triple_kind(edges, a, b, c):
    """'common effect', 'common cause' or 'causal chain' for consecutive nodes a – b – c of a path."""
    edge_set = set(edges)
    if (a, b) in edge_set and (c, b) in edge_set:
        return "common effect"
    if (b, a) in edge_set and (b, c) in edge_set:
        return "common cause"
    return "causal chain"


def triple_active(edges, a, b, c, evidence):
    """Slide 62: chains/forks are active iff b is unobserved; colliders iff b or a descendant is observed."""
    if triple_kind(edges, a, b, c) == "common effect":
        return b in evidence or bool(descendants_of(edges, b) & set(evidence))
    return b not in evidence


def path_report(edges, path, evidence):
    """(is the path active?, [(triple, kind, active), ...])."""
    triples = [tuple(path[i - 1:i + 2]) for i in range(1, len(path) - 1)]
    info = [(t, triple_kind(edges, *t), triple_active(edges, *t, evidence)) for t in triples]
    return all(active for _, _, active in info), info


def d_separated(edges, x, y, evidence=()):
    """True if every undirected path between x and y is blocked by `evidence` (slide 63)."""
    return not any(path_report(edges, p, set(evidence))[0] for p in undirected_paths(edges, x, y))

In [ ]:
# ── 4.3 First tests: the alarm network (slide 61 notes) ──
alarm_edges = edges_from_parents(ALARM_PARENTS)
for x, y, ev in [("B", "M", []), ("B", "M", ["A"]), ("B", "M", ["E", "A", "J"]),
                 ("B", "E", []), ("B", "E", ["A"]), ("B", "E", ["J"]), ("J", "M", ["A"])]:
    sep = d_separated(alarm_edges, x, y, ev)
    cmi = cond_mutual_info(alarm, [x], [y], ev)
    print(f"{x} ⊥ {y} | {{{', '.join(ev)}}}:  d-separated = {str(sep):5s}  exact I = {cmi:.2e} bits")

🔍 **What just happened?** The questions in the speaker notes of slide 61 — is $P(B,M)=P(B)P(M)$? is
$P(B,M\mid A)=P(B\mid A)P(M\mid A)$? is $P(B,M\mid E,A,J)=P(B\mid E,A,J)P(M\mid E,A,J)$? — get the answers *no*, *yes*, *yes*: the
only path $B\to A\to M$ is a chain, blocked as soon as $A$ is observed. $B$ and $E$ are d-separated with no evidence
(the collider $A$ blocks), but observing $A$ — or its descendant $J$ — opens it. In every case the graph's verdict
agrees with the exact conditional mutual information: d-separated ⇒ $I=0$; otherwise $I>0$.

✏️ **Your turn.** Decide by hand, then check with `d_separated(alarm_edges, ...)` and `cond_mutual_info`:
(a) $E\perp J\mid\{A\}$; (b) $E\perp J\mid\{M\}$; (c) $B\perp E\mid\{J, M\}$.

### 📑 Slide connection — `008_BN_mz_021825`, slides 61–63

- **What the slides show.** d-separation defines conditional independence between two variables: are X and Y independent given
    evidence {Z}? Yes if they are d-separated — consider all undirected paths, all inactive ⇒ independence (slide 61,
    with the query $U\perp V\mid\{W,X,Y\}$ on the 7-node graph). A path is inactive if it contains an inactive
    triple: causal chain or common cause with the middle node in {Z}; common effect with the middle node *and its
    descendants* not in {Z} (slide 62). Algorithm: list all paths, check each, all blocked ⇒ "guaranteed" independence
    (slide 63).
- **What this notebook adds.** A complete implementation (`undirected_paths`, `triple_kind`, `triple_active`, `path_report`,
    `d_separated`); the alarm-network questions of the slide-61 notes answered and checked against exact
    probabilities; a note on the clash between the evidence set {Z} and the node Z; the efficient Bayes-ball
    alternative.
- **Exam tip.** The collider rule is the classic trap: a collider is **active when observed** (or when a descendant is
    observed), the opposite of chains and forks. And "d-connected" only means "independence *not guaranteed*" —
    see Section 6.

---
## 5 · The 14 worked examples (slides 64–93)

> 📍 **Where we are.** The slides drill d-separation with 14 queries on one seven-node graph (adapted from a
> well-known lecture video). We reproduce all of them, draw them, and print the path-by-path reasoning.

The graph: $U\to W$, $V\to W$, $V\to X$, $V\to T$, $W\to Y$, $X\to Y$, $T\to Z$. Before running the next cell, try each
query yourself on paper: list the paths, classify each triple, decide.

| # | Query | # | Query |
|---|---|---|---|
| 1 | $V\perp Z\mid\{\}$ | 8 | $W\perp X\mid\{\}$ |
| 2 | $V\perp Z\mid\{T\}$ | 9 | $X\perp T\mid\{V\}$ |
| 3 | $U\perp V\mid\{\}$ | 10 | $X\perp W\mid\{U\}$ |
| 4 | $U\perp V\mid\{W\}$ | 11 | $Y\perp Z\mid\{\}$ |
| 5 | $U\perp V\mid\{X\}$ | 12 | $Y\perp Z\mid\{T\}$ |
| 6 | $U\perp V\mid\{Y\}$ | 13 | $Y\perp Z\mid\{X\}$ |
| 7 | $U\perp V\mid\{Z\}$ | 14 | $Y\perp Z\mid\{V\}$ |

In [ ]:
# ── 5.1 All 14 examples: our algorithm versus the slides' answers ──
DSEP_EDGES = [("U", "W"), ("V", "W"), ("V", "X"), ("V", "T"), ("W", "Y"), ("X", "Y"), ("T", "Z")]
DSEP_POS = {"U": (0.1, 0.9), "V": (0.5, 0.9), "W": (0.1, 0.5), "X": (0.5, 0.5), "T": (0.9, 0.5),
            "Y": (0.3, 0.1), "Z": (0.9, 0.1)}
EXAMPLES = [  # (number, X, Y, evidence, slides' verdict: True = independence guaranteed, slides)
    (1, "V", "Z", [], False, "64–65"), (2, "V", "Z", ["T"], True, "66–67"), (3, "U", "V", [], True, "68–70"),
    (4, "U", "V", ["W"], False, "71–72"), (5, "U", "V", ["X"], True, "73–74"), (6, "U", "V", ["Y"], False, "75–77"),
    (7, "U", "V", ["Z"], True, "78–79"), (8, "W", "X", [], False, "80–81"), (9, "X", "T", ["V"], True, "82–83"),
    (10, "X", "W", ["U"], False, "84–85"), (11, "Y", "Z", [], False, "86–87"), (12, "Y", "Z", ["T"], True, "88–89"),
    (13, "Y", "Z", ["X"], False, "90–91"), (14, "Y", "Z", ["V"], True, "92–93"),
]
rows = []
for k, x, y, ev, slide_says, slides in EXAMPLES:
    paths = undirected_paths(DSEP_EDGES, x, y)
    active = ["–".join(p) for p in paths if path_report(DSEP_EDGES, p, set(ev))[0]]
    ours = d_separated(DSEP_EDGES, x, y, ev)
    rows.append({"#": k, "query": f"{x} ⊥ {y} | {{{','.join(ev)}}}", "paths": len(paths),
                 "active paths": ", ".join(active) or "none",
                 "ours": "independent" if ours else "not guaranteed",
                 "slides": "independent" if slide_says else "not guaranteed",
                 "match": "✓" if ours == slide_says else "✗", "slides #": slides})
examples_table = pd.DataFrame(rows)
print(examples_table.to_string(index=False))
print(f"\nagreement with the slides: {(examples_table['match'] == '✓').sum()} / {len(EXAMPLES)}")

In [ ]:
# ── 5.2 The 14 examples at a glance (evidence in orange, query nodes in dashed red squares) ──
fig, axes = plt.subplots(2, 7, figsize=(12, 4.4))
for ax, (k, x, y, ev, _, _) in zip(axes.ravel(), EXAMPLES):
    sep = d_separated(DSEP_EDGES, x, y, ev)
    title = f"{k}: ${x} \\perp {y} \\mid \\{{{','.join(ev)}\\}}$\n" + ("independent" if sep else "not guaranteed")
    draw_graph(DSEP_EDGES, DSEP_POS, ax=ax, highlight=ev, radius=0.09)
    mark_nodes(ax, DSEP_POS, [x, y], half=0.12)
    ax.set_title(title, fontsize=8, color="tab:green" if sep else "tab:red")
plt.tight_layout(); plt.show()

🔍 **What just happened?** All 14 verdicts agree with the slides. The table lists every path and the active ones —
the reason for each verdict. Patterns to remember:

- **Examples 1–2, 11–12** — a causal chain through $T$ is open until $T$ is observed.
- **Examples 3–7** — $U$ and $V$ are connected only through the collider $W$ (directly, $U\to W\leftarrow V$, or via
  $U\to W\to Y\leftarrow X\leftarrow V$, which contains a second collider $Y$). Observing $W$ (Ex. 4) or its descendant $Y$
  (Ex. 6) opens the collider; observing $X$ (Ex. 5) or $Z$ (Ex. 7) does not — $Z$ is a descendant of $V$, not of $W$.
- **Examples 8–10, 14** — the common cause $V$ connects its children $W$, $X$, $T$ until $V$ is observed.
- **Example 13** — observing $X$ blocks one path ($Y\leftarrow X\leftarrow V$) but the other ($Y\leftarrow W\leftarrow V\to T\to Z$) stays
  open: **one** active path is enough for dependence.

⚠️ **A small slip on slide 81.** For Example 8 ($W\perp X$) the slide lists the paths "W—V—X" and "U—W—Y—X—V"; the
second is copied from Example 3. The second path between W and X is really **W—Y—X** (a collider at Y, blocked). The
verdict is unaffected because W—V—X is active ("do not care" about the others).

Now the slides' path-by-path explanations, generated by our code for four instructive examples.

In [ ]:
# ── 5.3 Path-by-path reasoning, in the style of the slides ──
def explain(edges, x, y, evidence):
    """Print every path between x and y with the status of each of its triples."""
    print(f"Query: {x} ⊥ {y} | {{{', '.join(evidence)}}}?")
    for path in undirected_paths(edges, x, y):
        active, info = path_report(edges, path, set(evidence))
        print(f"  path {'—'.join(path)}: {'ACTIVE' if active else 'inactive'}")
        for triple, kind, act in info:
            print(f"      {'—'.join(triple):8s} {'active  ' if act else 'inactive'} ({kind})")
    print("  Conclusion:", "independence guaranteed" if d_separated(edges, x, y, evidence)
          else "independence not guaranteed", "\n")


for k in (3, 6, 9, 13):
    _, x, y, ev, _, _ = EXAMPLES[k - 1]
    print(f"── Example {k}")
    explain(DSEP_EDGES, x, y, ev)

🔍 **What just happened?** The printout mirrors slides 70, 77, 83 and 91 line by line. In Example 3 both paths contain
an inactive collider. In Example 6 observing $Y$ activates both colliders at once ($W$ through its descendant $Y$, and
$Y$ itself), so both paths become active. Example 9 blocks the fork at $V$ on both paths (on the longer path the
collider at $Y$ is blocked as well). Example 13 shows one blocked and one active path.

✏️ **Your turn.** Predict, then check with `explain(DSEP_EDGES, ...)`: (a) $U\perp T\mid\{Y\}$; (b) $U\perp Z\mid\{W,V\}$;
(c) $W\perp Z\mid\{Y\}$; (d) the query on slide 61, $U\perp V\mid\{W,X,Y\}$.

### 📑 Slide connection — `008_BN_mz_021825`, slides 64–93

- **What the slides show.** Fourteen queries on the graph U → W ← V, V → X, V → T, W → Y ← X, T → Z, each with its paths, the status
    of each triple and the conclusion: (1) V⊥Z|{} not guaranteed; (2) V⊥Z|{T} guaranteed; (3) U⊥V|{} guaranteed;
    (4) U⊥V|{W} not guaranteed; (5) U⊥V|{X} guaranteed; (6) U⊥V|{Y} not guaranteed; (7) U⊥V|{Z} guaranteed;
    (8) W⊥X|{} not guaranteed; (9) X⊥T|{V} guaranteed; (10) X⊥W|{U} not guaranteed; (11) Y⊥Z|{} not guaranteed;
    (12) Y⊥Z|{T} guaranteed; (13) Y⊥Z|{X} not guaranteed; (14) Y⊥Z|{V} guaranteed. Once an active path is found the
    other paths are "do not care".
- **What this notebook adds.** All 14 reproduced by code, drawn in one figure, and explained path by path; a correction of the second
    path listed on slide 81; four extra queries for practice.
- **Exam tip.** Work systematically: (1) list *all* paths, ignoring directions; (2) for each path, classify every interior
    node as chain/fork/collider; (3) apply the rules, remembering descendants of colliders; (4) one active path ⇒
    not guaranteed. Stopping after the first blocked path is the most common mistake.

---
## 6 · Brute-force verification: does d-separation really mean independence?

> 📍 **Where we are.** d-separation is a statement about the *graph*. Its promise (slide 63): if $X$ and $Y$ are
> d-separated by the evidence, then $X\perp Y\mid\text{evidence}$ in **every** distribution that factorizes over the
> graph — whatever the CPT numbers. Let us test that promise by brute force.

We put **random** CPTs on the seven-node graph (each entry uniform between 0.05 and 0.95; SIMULATED), build the exact
$2^7=128$-row joint table, and compute the conditional mutual information for each of the 14 queries.

In [ ]:
# ── 6.1 Random CPTs on the 7-node graph: exact I(X;Y | evidence) for the 14 examples ──
DSEP_NODES = ["U", "V", "W", "X", "T", "Y", "Z"]
dsep_parents = {v: tuple(u for u, w in DSEP_EDGES if w == v) for v in DSEP_NODES}


def random_cpts(parents, rng, low=0.05, high=0.95):
    """ALARM-style CPTs with every entry drawn uniformly from [low, high]."""
    return {v: {vals: rng.uniform(low, high) for vals in itertools.product([0, 1], repeat=len(pa))}
            for v, pa in parents.items()}


rng = np.random.default_rng(SEED + 6)
rand_tab = JointTable(dsep_parents, random_cpts(dsep_parents, rng))
cmis = np.array([cond_mutual_info(rand_tab, [x], [y], ev) for _, x, y, ev, _, _ in EXAMPLES])
seps = np.array([d_separated(DSEP_EDGES, x, y, ev) for _, x, y, ev, _, _ in EXAMPLES])
for (k, x, y, ev, _, _), c, s in zip(EXAMPLES, cmis, seps):
    print(f"Ex {k:2d}  {x}⊥{y}|{{{','.join(ev)}}}  d-separated={str(s):5s}  I = {c:.2e} bits")

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(np.arange(1, 15), np.clip(np.abs(cmis), 1e-18, None), color=np.where(seps, "tab:green", "tab:red"))
ax.set_yscale("log"); ax.set_xticks(range(1, 15)); ax.set_xlabel("example")
ax.set_ylabel("I(X;Y | evidence) in bits (log)")
ax.set_title("random CPTs: d-separated (green) ⇒ I = 0 up to round-off; d-connected (red) ⇒ I > 0")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The green bars (d-separated queries) sit at the floating-point floor ($\lesssim10^{-16}$ bits,
i.e. exactly zero), the red bars (d-connected) many orders of magnitude higher. Fourteen queries with one set of CPTs
is a spot check. The next cell asks **every** possible question on this graph — all 21 pairs of nodes, each with all
32 evidence sets drawn from the other five nodes (672 queries) — for five independent random CPT sets.

In [ ]:
# ── 6.2 Exhaustive check: 672 queries × 5 random CPT sets ──
rng = np.random.default_rng(SEED + 60)
queries = [(x, y, ev) for x, y in itertools.combinations(DSEP_NODES, 2)
           for r in range(6) for ev in itertools.combinations([v for v in DSEP_NODES if v not in (x, y)], r)]
verdict = np.array([d_separated(DSEP_EDGES, x, y, ev) for x, y, ev in queries])
tables = [JointTable(dsep_parents, random_cpts(dsep_parents, rng)) for _ in range(5)]
cmi_all = np.array([[cond_mutual_info(tab, [x], [y], ev) for x, y, ev in queries] for tab in tables])
print(f"{len(queries)} queries: {verdict.sum()} d-separated, {(~verdict).sum()} d-connected")
print(f"largest |I| over the d-separated queries (5 CPT sets): {np.abs(cmi_all[:, verdict]).max():.1e} bits")
print(f"smallest I over the d-connected queries (5 CPT sets): {cmi_all[:, ~verdict].min():.1e} bits")

🔍 **What just happened?** Of the 672 queries, 240 are d-separated and 432 d-connected. Every d-separated query has
$I=0$ to machine precision in all five random networks (largest value $3\times10^{-16}$ bits): d-separation is
**sound** — it never claims an independence that does not hold. Every d-connected query showed some dependence, though
the weakest are tiny ($10^{-11}$ bits — still some 30,000 times above the round-off floor): dependence that
must travel along a long path through several noisy CPTs gets diluted. That is typical but **not guaranteed**:
d-connection means only "independence not guaranteed".

### 6.1 When a d-connected pair is independent: an incoherent feed-forward loop

A distribution is called **faithful** to its graph if *every* independence it has is implied by d-separation.
Random CPTs are faithful with probability 1, but special parameter values can create extra independences. A
biological example is the **incoherent feed-forward loop**, a common motif in gene regulation (Alon 2007): a
transcription factor $X$ activates a target $Y$ directly, and also induces a repressor $Z$ of the same target. If the
two routes cancel exactly, $X$ and $Y$ are uncorrelated although $X$ regulates $Y$. We simulate a linear-Gaussian
version (Module 15, Section 7.2): $Z=X+\varepsilon_Z$ ($X$ induces $Z$) and $Y=X-Z+\varepsilon_Y$ ($X$ activates $Y$,
$Z$ represses it), where the $\varepsilon$'s are independent standard normal noise terms.

In [ ]:
# ── 6.3 SIMULATED unfaithful circuit: X → Y directly and X → Z ⊣ Y, with exact cancellation ──
rng = np.random.default_rng(SEED + 63)
n = 100_000
X = rng.normal(size=n)
Z = 1.0 * X + rng.normal(size=n)           # X induces the repressor Z
Y = 1.0 * X - 1.0 * Z + rng.normal(size=n)  # X activates Y; Z represses Y — and Z carries +X, so the two routes cancel


def residual(a, b):
    """What is left of `a` after removing its best linear prediction from `b` (least squares, Module 06)."""
    B = np.column_stack([np.ones_like(b), b])
    return a - B @ np.linalg.lstsq(B, a, rcond=None)[0]


ffl_edges = [("X", "Y"), ("X", "Z"), ("Z", "Y")]
print("d-separated(X, Y | {}) :", d_separated(ffl_edges, "X", "Y", []))
print(f"corr(X, Y)            = {np.corrcoef(X, Y)[0, 1]:+.4f}   ← looks independent")
print(f"corr(X, Y | Z)        = {np.corrcoef(residual(X, Z), residual(Y, Z))[0, 1]:+.4f}   ← the direct effect reappears")

🔍 **What just happened?** $X$ and $Y$ are adjacent in the graph — they can never be d-separated — yet their
correlation is essentially zero, because the direct activation and the indirect repression cancel. Holding the
repressor fixed (residual correlation given $Z$) reveals the direct effect. Such exact cancellation needs finely tuned
parameters, which is why structure-learning methods *assume* faithfulness. Whether real circuits come close to
cancelling (incoherent feed-forward loops can produce adaptation, where a target returns to its baseline after a
stimulus) is an empirical question — one more reason to treat learned networks with care (Module 17).

✏️ **Your turn.** Weaken the induction of the repressor to `Z = 0.5 * X + ...` and re-run. What is corr(X, Y) now? For
$Y=aX+bZ+\varepsilon$ and $Z=cX+\varepsilon'$, show that $\operatorname{cov}(X,Y)=(a+bc)\operatorname{var}(X)$, so the
pair looks independent exactly when $a+bc=0$.

---
## 7 · Example 15, biology and real data

> 📍 **Where we are.** The slides end with a biological example (slides 94–95) and a summary (slide 96). We reproduce
> Example 15, show when its conclusion breaks, and then do what the slides do not: test d-separation predictions of
> a real signalling network on real single-cell measurements.

**Example 15.** In the regulatory network of slides 94–95, two TF genes are regulated by the same transcription
factor. Are they independent given that factor? The path between them is a common cause, blocked when the factor is
observed: **independent** (if there is no other active path). In our network of Section 3, G1 and G2 share the
regulator R1 — and also the target T2.

In [ ]:
# ── 7.1 Example 15 on the regulatory network of Section 3 ──
for ev in [[], ["R1"], ["R1", "T2"], ["T2"]]:
    sep = d_separated(GRN_EDGES, "G1", "G2", ev)
    print(f"G1 ⊥ G2 | {{{', '.join(ev)}}}: {'independent' if sep else 'not guaranteed':15s}   "
          f"exact I (random CPTs) = {cond_mutual_info(grn, ['G1'], ['G2'], ev):.1e} bits")

🔍 **What just happened?** Without evidence, G1 and G2 are dependent through their common regulator R1. Observing R1
blocks that path, and the collider at their shared target T2 is blocked as long as T2 is not observed: independent,
as on slide 95. But measure the shared target T2 as well, and the two TF genes become dependent again (explaining
away: if T2 is on and G1 is off, G2 is probably on).

✏️ **Your turn.** Suppose R2 also regulates G1: add `("R2", "G1")` to a copy of `GRN_EDGES`. Is
$G1\perp G2\mid\{R1\}$ still true? Which evidence set makes the two TF genes independent again?

**What this means for a biologist.** Two co-expressed genes are *not* evidence that one regulates the other: a shared
regulator predicts exactly this correlation, and predicts that it disappears once the regulator's activity is held
fixed. The prediction holds only if the graph is right. If an **unmeasured** second regulator also controls both genes,
conditioning on R1 leaves a residual correlation — the graph we drew was missing a common cause.

### 7.1 Testing two predictions of the Sachs network on real cells

d-separation turns a network into testable predictions. In the consensus network of Sachs *et al.* (2005):
- p38 and JNK both have exactly the parents PKA and PKC and no children, so every path between them passes through
  PKA or PKC as a chain or fork ⇒ **p38 ⊥ JNK | {PKA, PKC}**;
- Raf → Mek → Erk, plus PKA → Raf and PKA → Erk ⇒ **Raf ⊥ Erk | {Mek, PKA}**.

Let the code confirm the d-separation claims, then test them on the 1,800 unperturbed cells. With finite data a plug-in
estimate of $I$ is never exactly 0 (it is biased upwards), so we compare it with a **permutation null**: shuffle one
protein's values *within* each group of cells that share the same conditioning values. This destroys any dependence
given the conditioning set while keeping everything else, and shows how large $I$ gets by chance alone. The
**p-value** is the fraction of shuffled data sets whose $I$ is at least as large as the observed one (counting the
observed data set itself, so with 500 shuffles the smallest possible p-value is $1/501\approx0.002$); a small p-value
means "chance alone rarely produces dependence this strong" (Module 06 uses p-values for GWAS).

In [ ]:
# ── 7.2 Real data: plug-in conditional mutual information with a stratified permutation test ──
sachs = load_sachs("discrete")
data = sachs["data"]
print(data.shape, "|", sachs["source"])
print("cells per level — PKA:", np.bincount(data["pka"]), "| PKC:", np.bincount(data["pkc"]))


def cmi_from_counts(x, y, strata):
    """Plug-in I(X;Y|S) in bits from integer arrays (strata = one integer label per cell)."""
    counts = np.zeros((strata.max() + 1, x.max() + 1, y.max() + 1))
    np.add.at(counts, (strata, x, y), 1)
    p = counts / counts.sum()
    ps, pxs, pys = p.sum(axis=(1, 2)), p.sum(axis=2), p.sum(axis=1)
    keep = p > 0
    ratio = p * ps[:, None, None] / (pxs[:, :, None] * pys[:, None, :] + 1e-300)
    return float(np.sum(p[keep] * np.log2(ratio[keep])))


def permutation_test(df, x, y, given, n_perm, rng):
    """Observed I(x;y|given) and its p-value when x is shuffled within each stratum of `given`."""
    strata = (np.unique(df[given].to_numpy(), axis=0, return_inverse=True)[1].ravel() if given
              else np.zeros(len(df), int))
    xv, yv = df[x].to_numpy(), df[y].to_numpy()
    observed = cmi_from_counts(xv, yv, strata)
    groups = [np.flatnonzero(strata == s) for s in np.unique(strata)]
    null = np.empty(n_perm)
    for b in range(n_perm):
        xp = xv.copy()
        for g in groups:
            xp[g] = rng.permutation(xv[g])
        null[b] = cmi_from_counts(xp, yv, strata)
    return observed, null, (1 + np.sum(null >= observed)) / (1 + n_perm)

In [ ]:
# ── 7.3 Do the real cells obey the consensus network's d-separation statements? ──
rng = np.random.default_rng(SEED + 7)
tests = [("p38", "jnk", []), ("p38", "jnk", ["pka", "pkc"]), ("raf", "erk", []), ("raf", "erk", ["mek", "pka"])]
results = []
for x, y, given in tests:
    obs, null, pval = permutation_test(data, x, y, given, 500, rng)
    results.append((x, y, given, obs, null, pval))
    print(f"{x} ⊥ {y} | {{{', '.join(given)}}}: graph: {'d-separated' if d_separated(SACHS_CONSENSUS_EDGES, x, y, given) else 'd-connected'}"
          f" | data: I = {obs:.4f} bits, null 95th percentile = {np.quantile(null, 0.95):.4f}, p = {pval:.3f}")

fig, axes = plt.subplots(1, 4, figsize=(12, 2.8))
for ax, (x, y, given, obs, null, pval) in zip(axes, results):
    ax.hist(null, bins=30, color="0.7")
    ax.axvline(obs, color="tab:red", lw=2)
    ax.set_title(f"{x} ⊥ {y} | {{{','.join(given)}}}\np = {pval:.3f}", fontsize=9)
    ax.set_xlabel("I (bits)")
axes[0].set_ylabel("permutations")
fig.suptitle("real Sachs data: observed conditional MI (red) vs permutation null (grey)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** (Check the printed numbers above; they come from the real data.) The code first confirms the
graph's claims: p38 and JNK, and Raf and Erk, are *not* d-separated with empty evidence but *are* d-separated by
{PKA, PKC} and {Mek, PKA}, respectively. The data then tell two different stories:

- **p38 and JNK** are clearly dependent in the cells ($p=0.002$, the smallest possible with 500 permutations) — and
  conditioning on PKA and PKC does **not** remove the dependence: the conditional MI is *larger* than the marginal one
  and far outside the permutation null. **The data reject this prediction of the consensus network.**
- **Raf and Erk** show no detectable dependence even *without* conditioning ($p\approx0.2$–$0.3$), so the conditional
  test has nothing to remove. The data are *consistent* with the prediction, but this is weak support: the test also
  fails to see the dependence that the graph says should be there marginally (low power).

Why could a curated network fail a test on its own data? Several honest explanations, none of which we can settle
here: (1) **discretisation** — conditioning on a three-level version of PKA does not hold the real, continuous PKA
activity fixed, so information leaks through; PKA is almost constant in these cells (1,703 of 1,800 in the middle
level), which makes it a poor conditioning variable; (2) **unmeasured common regulators** of p38 and JNK (the
upstream MAP kinase kinases and their activators, some of which act on both pathways, were not measured) act as hidden
common causes; (3) the consensus network is a summary of
literature and of Sachs *et al.*'s analysis, **not** a complete wiring diagram; (4) cells are heterogeneous, and
mixing cell states can create dependences that hold in no single state. A d-separation statement is a falsifiable
prediction — and testing it is how networks get improved.

⚠️ Each test here is one of many one could run; with dozens of tests, some "significant" results arise by chance
(multiple testing, Module 23). The p38/JNK result is far from borderline, but in a systematic analysis you would
correct for the number of tests.

### 📑 Slide connection — `008_BN_mz_021825`, slides 94–96

- **What the slides show.** Example 15 (slides 94–95): in the TF → TF-gene → target network, two TF genes regulated by the same TF
    are independent given that TF — "Inactive (common cause blocked)". Summary (slide 96): a variable is
    conditionally independent of all others given its (minimal) Markov blanket (parents, children, co-parents);
    d-separation: X and Y are conditionally independent given {Z} if all paths are inactive, using the three
    triple rules.
- **What this notebook adds.** Example 15 recomputed, plus the case the slide does not show (a shared observed target re-opens the path);
    an unfaithful incoherent feed-forward loop; a stratified permutation test of two d-separation predictions of the
    Sachs consensus network on real single-cell data, one of which fails — with possible reasons.
- **Exam tip.** When an exam question gives a biological network, translate it into triples first: shared regulator =
    common cause (blocked by observing it), shared target = common effect (opened by observing it or its
    descendants), cascade = causal chain.

---
## 📌 Key takeaways

- $A\perp B\mid C$ means $P(A\mid B,C)=P(A\mid C)$ for **all** values; the conditional mutual information is 0 exactly
  in that case. Independence and conditional independence do not imply each other.
- **Three triples:** causal chain and common cause are blocked by observing the middle node; a common effect
  (collider) is blocked *unless* the middle node or one of its descendants is observed — explaining away
  ($P(B{=}1\mid A{=}1)\approx0.32$ drops to $\approx0.09$ when the earthquake is known) and collider bias in case-only studies.
- **Markov blanket** = parents + children + co-parents; given its blanket a node is independent of all other nodes
  (B: {A, E}; A: {B, E, J, M}; J: {A}).
- **d-separation:** list all undirected paths, classify every triple, one inactive triple blocks a path; all paths
  blocked ⇒ independence **guaranteed** in every distribution that factorizes over the graph. All 14 slide examples
  and 672 brute-force queries confirm it.
- d-connected only means independence is **not guaranteed**; special parameter values (e.g. a perfectly cancelling
  incoherent feed-forward loop) can hide a dependence — the faithfulness assumption.
- A network's d-separation statements are **testable predictions**. On real Sachs cells, p38 ⊥ JNK | {PKA, PKC} is
  clearly rejected — a reminder that a curated graph, and any learned one, is a hypothesis.

---
## ✅ Checkpoint

1. In the alarm network, are J and M independent? Are they independent given A? Given B?
   <details><summary>Answer</summary>Not independent (common cause A, active). Independent given A (the fork is blocked). Not independent given B: the path J←A→M is still active because A is unobserved.</details>
2. In the network $X\to B\leftarrow Y$, $B\to C$, which of these hold: $X\perp Y$; $X\perp Y\mid B$; $X\perp Y\mid C$?
   <details><summary>Answer</summary>Only $X\perp Y$. Observing the collider B, or its descendant C, opens the path (explaining away).</details>
3. Write the Markov blanket of V and of W in the seven-node graph of the examples.
   <details><summary>Answer</summary>V: children W, X, T and co-parent U (other parent of W) → {U, W, X, T}. W: parents U, V; child Y; co-parent X (other parent of Y) → {U, V, X, Y}.</details>
4. Decide $U\perp X\mid\{Y\}$ in the seven-node graph, listing the paths.
   <details><summary>Answer</summary>Paths: U–W–V–X (collider W; its descendant Y is observed → active; V fork unobserved → active) and U–W–Y–X (W chain unobserved → active; Y collider observed → active). Both active → independence not guaranteed.</details>
5. Using the slide CPTs, compute $P(B{=}1\mid A{=}1,E{=}1)$ by hand.
   <details><summary>Answer</summary>$\frac{P(B)P(A\mid B,E)}{P(B)P(A\mid B,E)+P(\neg B)P(A\mid\neg B,E)}=\frac{0.05\cdot0.95}{0.05\cdot0.95+0.95\cdot0.5}=\frac{0.0475}{0.5225}\approx0.091$ (P(E) cancels).</details>
6. Code reading: in `triple_active`, why does the collider branch look at `descendants_of(edges, b)` while the other branch does not?
   <details><summary>Answer</summary>Chains and forks are blocked only by observing the middle node itself. A collider is opened by observing the middle node *or any descendant*, because a descendant carries information about the collider's value.</details>
7. A plug-in conditional mutual information on 1,800 cells is 0.004 bits. Is that evidence of dependence?
   <details><summary>Answer</summary>Not by itself: plug-in estimates are biased upwards and never exactly 0. Compare with a null distribution (e.g. stratified permutations); only values far in its tail indicate dependence.</details>
8. Give one biological reason why two genes regulated by the same measured TF might still be correlated given that TF.
   <details><summary>Answer</summary>An unmeasured second common regulator (hidden common cause); a direct interaction between the genes; or the TF measurement is coarse/noisy (e.g. discretised) so conditioning on it does not fully block the path.</details>

## 🧠 Questions & discussion

1. Collider bias appears whenever a study selects on an outcome (hospital patients, cells that survived a drug screen, sequenced tumours). Describe one example from your field and draw its graph.
2. The Markov blanket of a protein is the ideal feature set to predict it. Why might a lasso regression (Module 07) on all other proteins select a different set in practice?
3. Faithfulness fails for finely tuned parameters. Biological systems are tuned by evolution. Is faithfulness a reasonable assumption for gene regulatory networks? Argue both sides.
4. Our test of p38 ⊥ JNK | {PKA, PKC} used discretised data. Design a better test with the continuous measurements (Module 18 will help). What would you expect to change?
5. d-separation needs the whole graph. If you are unsure about one edge, how could you use d-separation to design an experiment that distinguishes the two graphs?

## 🚀 Bonus — going further

- **Bayes-ball.** Implement the linear-time reachability algorithm for d-separation (Koller & Friedman 2009, Algorithm 3.1) and check that it agrees with `d_separated` on the 672 queries of Section 6.
- **All independences of a graph.** For the alarm network, list every statement $X\perp Y\mid S$ implied by d-separation. Which of them does the exact joint satisfy? (All of them, and — with the slide CPTs — no others?)
- **The PC algorithm in miniature.** Starting from a complete undirected graph on the seven nodes, delete an edge X–Y whenever some evidence set makes $I(X;Y\mid S)=0$ in the exact joint of Section 6. Do you recover the skeleton of the true graph? This is the core of constraint-based structure learning.
- **Related notebooks in this repository.** [`../../GenerativeDiscriminativeClassification/`](../../GenerativeDiscriminativeClassification/) (naïve Bayes: a graph with one strong independence assumption, tested on real data) and [`../../FundamentalsLearningUncertainty/`](../../FundamentalsLearningUncertainty/) (probability refresher).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 16 — Bayesian networks II: independence           date:
Input:            alarm network (slide CPTs); 7-node example graph with random CPTs; 12-node TF network;
                  Sachs 2005 consensus graph + 1,800 real unperturbed cells (discretised)
Output:           verdicts for the 14 slide examples (14/14 agree), 672-query brute-force check,
                  explaining-away probabilities, permutation tests of two network predictions
Method:           exact joint tables; conditional mutual information; Markov blanket; d-separation by path
                  enumeration; stratified permutation test
Key equation:     X ⊥ Y | Z  ⇔  P(x | y, z) = P(x | z) for all values  ⇔  I(X;Y|Z) = 0
Key parameters:   evidence set; CPT values (strength, not existence, of dependence); number of permutations
What changed when I modified the evidence / the CPTs / the conditioning set:
One failure mode: an unmeasured common cause or a discretised conditioning variable leaves dependence
Connection to 02-620 slides: 008_BN_mz_021825 slides 38–96
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, deck `008_BN_mz_021825` (course design S. Kim; instructors M. Xu and M. Zhang), slides 38–96.
- Koller D., Friedman N. (2009) *Probabilistic Graphical Models: Principles and Techniques*. MIT Press — Chapter 3 (independencies, d-separation, the reachability algorithm).
- Shachter R. D. (1998) Bayes-Ball: the rational pastime (for determining irrelevance and requisite information in belief networks and influence diagrams). *Proceedings of the 14th Conference on Uncertainty in Artificial Intelligence (UAI)*, 480–487.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — §10.5 (conditional independence properties of DGMs).
- Pearl J. (1988) *Probabilistic Reasoning in Intelligent Systems*. Morgan Kaufmann.
- Geiger D., Verma T., Pearl J. (1990) Identifying independence in Bayesian networks. *Networks* 20:507–534.
- Alon U. (2007) Network motifs: theory and experimental approaches. *Nature Reviews Genetics* 8:450–461.
- Sachs K., Perez O., Pe'er D., Lauffenburger D. A., Nolan G. P. (2005) Causal protein-signaling networks derived from multiparameter single-cell data. *Science* 308:523–529.

## ➡️ Next

You can now read independence off a graph. In
[Module 17 · Bayesian networks III: inference and learning a signalling network](17_bayesian_networks_inference_learning.ipynb)
you will compute with the network — enumeration, variable elimination, sampling — and then learn CPTs and whole
graphs (Chow–Liu trees, BIC hill climbing) from the real Sachs data, including what observational data cannot
tell you about arrow directions.